# Delayed Fractional-Order Graph Dynamics for Integrated Modular Avionics
### Simulation & Reproducibility Model — Google Colab notebook

Reproduces the numerical scenarios and **Figures 1–9** of the manuscript
*"Delayed Fractional-Order Graph Dynamics for Cascade Escalation and Reconfiguration
Failure in Integrated Modular Avionics"*.

> ⚠️ **Synthetic research model.** All parameters are synthetic / mechanism-oriented;
> they are **not** from any real Airbus/Boeing/certified IMA platform and the model is
> **not** certification software.

Run the cells top-to-bottom (`Runtime ▸ Run all`). The full ensemble is reduced by
default so the notebook finishes in a few minutes on a free Colab CPU instance.


## 1. Setup — dependencies and package


In [ ]:
# Install open-source dependencies (already present on most Colab images).
!pip -q install numpy scipy pandas matplotlib networkx scikit-learn pyyaml pytest

In [ ]:
# Make the package importable. If this notebook is opened inside the cloned
# repository, it is used directly; otherwise clone it from GitHub.
import os, sys, pathlib

REPO_URL = 'https://github.com/omega2417/bnt'   # <-- repository hosting this package
SUBDIR   = 'ima_fractional_model'

def find_pkg():
    for base in ['.', SUBDIR, f'bnt/{SUBDIR}']:
        if pathlib.Path(base, 'src', 'graph_model.py').exists():
            return str(pathlib.Path(base).resolve())
    return None

root = find_pkg()
if root is None:
    # running on a bare Colab: clone the repository
    !git clone --depth 1 {REPO_URL} bnt 2>/dev/null
    root = find_pkg()
assert root, 'Could not locate the ima_fractional_model package.'
os.chdir(root); sys.path.insert(0, root)
print('package root:', root)

## 2. Load the synthetic 22-node IMA architecture (Figure 1)


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src import load_graph, ScenarioParams, cascade_threshold
from src.scenarios import run_scenario
from src import visualization as viz

graph = load_graph('config/ima_22node.yaml')
print(f'nodes = {graph.n}  (CPM/SW/RDC = 8/4/10)')
print('DAL-A nodes:', [graph.node_ids[i] for i in graph.dal_A_idx])

viz.figure1_topology(graph, 'figures')
from IPython.display import Image, display
display(Image('figures/figure_1.png'))

## 3. Cascade thresholds $R_c$ (Equation 7)

`R_c < 1` = analytically certified stable regime;  `R_c ≥ 1` = cascade-prone
(the sufficient stability certificate is unavailable — *not* an automatic catastrophe).


In [ ]:
scen = {'S1': (0.93, 0.55), 'S2': (1.60, 0.75), 'S3': (1.60, 0.38)}
for name,(ks,xi) in scen.items():
    tr = cascade_threshold(graph, ScenarioParams(ks, xi, 0.8, 0.8))
    print(f'{name}:  Rc = {tr.Rc:.3f}   {tr.regime}')

## 4. Scenarios S1–S3 — absorption, containment, catastrophe (Figures 3–5)


In [ ]:
sims, tcat = {}, {}
for name,(ks,xi) in scen.items():
    T = 200 if name=='S1' else 400
    r = run_scenario(graph, ScenarioParams(ks, xi, 0.8, 0.8), name=name,
                     seed_node='RDC1', seed_magnitude=0.55, T=T, h=0.05, x_cat=0.6)
    r.sim._Rc = f'{r.Rc:.2f}'
    sims[name], tcat[name] = r.sim, r.T_cat
    print(f'{name}:  catastrophe={r.catastrophe}  T_cat={r.T_cat}  '
          f'cascade={r.terminal_cascade_size*100:.0f}%  max DAL-A={r.max_dalA:.3f}')

In [ ]:
viz.figure3_trajectories(graph, sims, 0.6, tcat, 'figures')
display(Image('figures/figure_3.png'))

In [ ]:
viz.figure4_backlog_heatmap(graph, sims['S3'], tcat['S3'], 'figures')
viz.figure5_cascade_graph(graph, sims['S3'], 'figures')
display(Image('figures/figure_4.png'))
display(Image('figures/figure_5.png'))

## 5. Delay & memory — critical delay $\tau^*(\alpha)$ (Theorem 4, Figure 6)

Stronger hereditary memory (smaller $\alpha$) enlarges the delay-stability region;
the integer-order model ($\alpha=1$) is the most conservative.


In [ ]:
from src import critical_delay_scalar
from src.scenarios import delay_memory_grid
alphas=[0.6,0.7,0.8,0.9,1.0]
taus=[critical_delay_scalar(a)[0] for a in alphas]
for a,t in zip(alphas,taus): print(f'tau*({a}) = {t:.3f}')
grid_taus=np.linspace(0.5,2.5,5)
grid=delay_memory_grid(graph, alphas, grid_taus, xi=0.64, T=250)  # reduced for speed
viz.figure6_delay_memory(alphas, taus, alphas, grid_taus, grid, 'figures')
display(Image('figures/figure_6.png'))

## 6. Mean-field bistability (Theorem 3, Figure 7)


In [ ]:
from src.scenarios import mean_field_trajectory, mean_field_fixed_points, MEANFIELD
P=dict(MEANFIELD); P['xi']=0.50
traj=[]
for x0 in np.linspace(0,1,7):
    for q0 in np.linspace(0.2,2.0,7):
        _,X,Q=mean_field_trajectory(x0,q0,T=120,P=P)
        traj.append((X,Q,'CAT' if X[-1]>0.5 else 'NOM'))
fps=[(f[0],f[1],'stable') for f in mean_field_fixed_points(P) if f[0]<0.3]
_,Xc,Qc=mean_field_trajectory(0.9,1.8,T=200,P=P)
fps.append((float(Xc[-1]),float(Qc[-1]),'stable'))
viz.figure7_phase_portrait(traj, fps, 'figures')
display(Image('figures/figure_7.png'))

## 7. Reconfiguration tipping & priority-conflict sensitivity (Figure 8)


In [ ]:
from src.scenarios import tipping_sweep, priority_sweep
tip=tipping_sweep(graph, np.linspace(0.35,0.75,9), T=300)
pri=priority_sweep(graph, np.linspace(0.4,2.0,7), T=300)
cat_xi=[r['xi'] for r in tip if r['catastrophe']]
xi_star=max(cat_xi) if cat_xi else tip[0]['xi']
viz.figure8_tipping(tip, pri, xi_star, 'figures')
display(Image('figures/figure_8.png'))

## 8. Ensemble uncertainty analysis (Figure 9, Table 2)

Latin-Hypercube ensemble with right-censored catastrophe statistics
(Wilson CI, Kaplan–Meier median). `N` is reduced for Colab; use `N=100` to match the paper.


In [ ]:
from src.sensitivity import run_ensemble, summarize_ensemble, ensemble_prcc, PRCC_INPUTS
import pandas as pd
N = 30   # increase to 100 to reproduce Table 2 exactly (slower)
summary={}
for name,(ks,xi) in scen.items():
    runs=run_ensemble(graph, name=name, kappa_scale=ks, xi=xi, N=N, T=400, seed=12345)
    s=summarize_ensemble(runs, 400.0); summary[name]=s
    print(f"{name}: P_cat={s['p_cat']:.2f} "
          f"[{s['wilson_low']:.3f}, {s['wilson_high']:.3f}]  KM median T_cat={s['km_median']}")
    if name=='S3':
        prcc=ensemble_prcc(runs, PRCC_INPUTS, 400.0)
        print('  PRCC(T_cat):', {k: round(v['prcc'],2) for k,v in prcc.items()})
fig_sum={n:dict(p_cat=summary[n]['p_cat'], wilson_low=summary[n]['wilson_low'],
                wilson_high=summary[n]['wilson_high'], km_median=summary[n]['km_median'])
         for n in scen}
viz.figure9_ensemble(fig_sum, 'figures')
display(Image('figures/figure_9.png'))

## 9. One-command full reproduction (optional)

Regenerates **every** figure and table into `figures/` and `results/`.
Full run ≈ 12–18 min; `--fast` ≈ 3–4 min.


In [ ]:
# !python scripts/reproduce_all.py --fast
print('Run the line above (remove the # ) to reproduce the entire package.')

---
Model structure (Figure 2):


In [ ]:
viz.figure2_model_structure('figures')
display(Image('figures/figure_2.png'))